# Fase 4 — Modelo Preditivo: LightGBM + SHAP

**Objetivo:** Prever se ocorrerá um evento Don't Go nos próximos 60 minutos, com antecedência suficiente para acionar manutenção preventiva.

**Split temporal:**
- Treino: Jan–Abr/2025
- Validação: Mai/2025 (seleção de threshold)
- Teste: Jun/2025 (avaliação final)

**Meta:** F1 ≥ 0.75 no conjunto de teste.

In [ ]:
import sys
sys.path.insert(0, '../src')

import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

from models import run_pipeline, evaluate_model, find_best_threshold, compute_shap_values, load_model
from features import get_feature_columns

GOLD_DIR = Path('../outputs/gold')
FIGURES_DIR = Path('../outputs/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Executar Pipeline Completo

In [ ]:
# run_pipeline carrega gold (ou gera se necessário), treina LightGBM,
# otimiza threshold no val set, avalia no test set e calcula SHAP.
results = run_pipeline(rebuild_gold=False)

In [ ]:
model       = results['model']
metrics     = results['metrics']
shap_vals   = results['shap_values']
X_shap      = results['X_shap']
feature_cols = results['feature_cols']
threshold   = results['threshold']

print(f'\nThreshold ótimo (val): {threshold:.2f}')
print(f'F1-Score  (teste):  {metrics["f1"]:.4f}')
print(f'Precision (teste):  {metrics["precision"]:.4f}')
print(f'Recall    (teste):  {metrics["recall"]:.4f}')
print(f'ROC-AUC   (teste):  {metrics["roc_auc"]:.4f}')
print(f'PR-AUC    (teste):  {metrics["pr_auc"]:.4f}')

## 2. Curva ROC e Curva Precision-Recall

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

df_gold = results['df']
feature_cols_list = get_feature_columns(df_gold)

mask_test = df_gold['Data_Evento'].dt.month().is_in([6])
X_test = df_gold.filter(mask_test).select(feature_cols_list).to_pandas()
y_test = df_gold.filter(mask_test)['is_dont_go_next_60m'].cast(pl.Int8).to_pandas()

y_prob = model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
prec, rec, _ = precision_recall_curve(y_test, y_prob)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name=f'ROC (AUC={metrics["roc_auc"]:.3f})'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', line=dict(dash='dash', color='gray'), name='Aleatório'))
fig.update_layout(
    title='Curva ROC — Conjunto de Teste (Jun/2025)',
    xaxis_title='Taxa de Falsos Positivos',
    yaxis_title='Taxa de Verdadeiros Positivos',
    width=650, height=500,
)
fig.show()
fig.write_image(str(FIGURES_DIR / 'roc_curve.png'))

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=rec, y=prec, mode='lines', name=f'PR (AUC={metrics["pr_auc"]:.3f})'))
fig2.update_layout(
    title='Curva Precision-Recall — Conjunto de Teste (Jun/2025)',
    xaxis_title='Recall',
    yaxis_title='Precision',
    width=650, height=500,
)
fig2.show()
fig2.write_image(str(FIGURES_DIR / 'pr_curve.png'))

## 3. Matriz de Confusão

In [ ]:
cm = np.array(metrics['confusion_matrix'])
labels = ['Sem DG', 'Com DG']

fig3 = px.imshow(
    cm,
    x=labels, y=labels,
    text_auto=True,
    title=f'Matriz de Confusão (threshold={threshold:.2f})',
    labels=dict(x='Predito', y='Real'),
    color_continuous_scale='Blues',
)
fig3.update_layout(width=450, height=400)
fig3.show()
fig3.write_image(str(FIGURES_DIR / 'confusion_matrix.png'))

## 4. Importância das Features (SHAP)

In [ ]:
mean_abs_shap = pd.Series(
    np.abs(shap_vals).mean(axis=0),
    index=X_shap.columns,
    name='mean_abs_shap',
).sort_values(ascending=False)

top20 = mean_abs_shap.head(20).reset_index()
top20.columns = ['feature', 'mean_abs_shap']

fig4 = px.bar(
    top20,
    x='mean_abs_shap', y='feature',
    orientation='h',
    title='Top-20 Features — Importância SHAP (|valor médio|)',
    labels={'mean_abs_shap': 'Mean |SHAP|', 'feature': ''},
)
fig4.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig4.show()
fig4.write_image(str(FIGURES_DIR / 'shap_importance.png'))

In [ ]:
# SHAP beeswarm-style: scatter dos valores SHAP vs feature value (top-10)
top10_features = mean_abs_shap.head(10).index.tolist()

for feat in top10_features[:5]:  # primeiras 5 para não sobrecarregar o notebook
    feat_idx = list(X_shap.columns).index(feat)
    shap_col = shap_vals[:, feat_idx]
    feat_values = X_shap[feat].values

    fig_s = px.scatter(
        x=feat_values, y=shap_col,
        opacity=0.3,
        labels={'x': feat, 'y': 'SHAP value'},
        title=f'SHAP: {feat}',
        trendline='lowess',
    )
    fig_s.show()

## 5. Análise de Probabilidades por Equipamento (Jun/2025)

In [ ]:
# Timeline de probabilidade para os 5 equipamentos com mais eventos DG no teste
df_test = df_gold.filter(mask_test).to_pandas()
df_test['prob_dg'] = model.predict_proba(X_test)[:, 1]
df_test['pred_dg'] = (df_test['prob_dg'] >= threshold).astype(int)

top_tags = (
    df_test[df_test['is_dont_go_next_60m'] == 1]
    .groupby('TAG')['is_dont_go_next_60m']
    .sum()
    .nlargest(5)
    .index.tolist()
)
print('Top-5 equipamentos com mais eventos DG em Jun:')
print(top_tags)

In [ ]:
tag = top_tags[0]  # equipamento mais crítico
df_tag = df_test[df_test['TAG'] == tag].sort_values('Data_Evento')

fig5 = go.Figure()
fig5.add_trace(go.Scatter(
    x=df_tag['Data_Evento'], y=df_tag['prob_dg'],
    mode='lines', name='P(DG nos próximos 60m)', line=dict(color='royalblue')
))
fig5.add_trace(go.Scatter(
    x=df_tag[df_tag['is_dont_go_next_60m'] == 1]['Data_Evento'],
    y=df_tag[df_tag['is_dont_go_next_60m'] == 1]['prob_dg'],
    mode='markers', name='Real: pré-DG',
    marker=dict(color='red', size=5, symbol='x')
))
fig5.add_hline(y=threshold, line_dash='dash', line_color='orange', annotation_text=f'Threshold={threshold:.2f}')
fig5.update_layout(
    title=f'Timeline de probabilidade — {tag} (Jun/2025)',
    xaxis_title='Data/Hora',
    yaxis_title='Probabilidade',
    height=450,
)
fig5.show()
fig5.write_image(str(FIGURES_DIR / f'timeline_{tag.replace("/","_")}.png'))

## 6. Importância de Features (LightGBM nativa)

In [ ]:
import lightgbm as lgb

lgb_imp = pd.DataFrame({
    'feature': model.feature_name_,
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'split': model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False).head(20)

fig6 = px.bar(
    lgb_imp,
    x='gain', y='feature',
    orientation='h',
    title='Top-20 Features — Importância por Ganho (LightGBM)',
    labels={'gain': 'Ganho total', 'feature': ''},
)
fig6.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig6.show()
fig6.write_image(str(FIGURES_DIR / 'lgbm_importance.png'))

## 7. Resumo Final

In [ ]:
cm = np.array(metrics['confusion_matrix'])
tn, fp, fn, tp = cm.ravel()

print('='*50)
print('    RESULTADO FINAL — Conjunto de Teste (Jun/2025)')
print('='*50)
print(f'  Threshold ótimo  : {threshold:.2f}')
print(f'  F1-Score         : {metrics["f1"]:.4f}  (meta: ≥ 0.75)')
print(f'  Precision        : {metrics["precision"]:.4f}')
print(f'  Recall           : {metrics["recall"]:.4f}')
print(f'  ROC-AUC          : {metrics["roc_auc"]:.4f}')
print(f'  PR-AUC           : {metrics["pr_auc"]:.4f}')
print(f'\n  Verdadeiros Pos  : {tp:,}  (DG detectados)')
print(f'  Falsos Positivos : {fp:,}  (alarmes falsos)')
print(f'  Falsos Negativos : {fn:,}  (DG perdidos)')
print(f'  Verdadeiros Neg  : {tn:,}')
print(f'\n  Meta atingida    : {"SIM ✓" if metrics["f1"] >= 0.75 else "NÃO — revisar features/params"}')
print('='*50)